# crisis detector and adaptive threshold

Idea behind calculating the crisis score and adapting fake/real classification threshold accordingly:

def calculate_crisis_score(news_stream):
    #Input: Real-time news headlines + metadata
    #Output: Crisis score (0-1), Crisis type

    signals = {
        'volume_spike': detect_unusual_volume(news_stream),
        'sentiment_shift': track_sentiment_over_time(news_stream),
        'keyword_clustering': detect_crisis_keywords(['election','pandemic','war']),
        'source_diversity': are many sources reporting the same event?
    }
    
    crisis_score = weighted_average(signals)  ***#Need to figure out how to weigh each signal's effect on the crisis score***
    
    ***#Could put in more conditionals for higher risk, more moderate cirsis modes.***
    if crisis_score > 0.7:
        # ENTER CRISIS MODE
        adaptive_threshold = 0.3  # Stricter (catch more fake news)
    else:
        adaptive_threshold = 0.5  # Normal mode 
        
    return crisis_score, adaptive_threshold

# Notes for improvement: 

# Crisis keywords by category. 
        # Hardcoding keywords is a starting point but has serious limitations 
        # especially if it comes to crises not seen before, keywords become outdated. 
        # In the initial run it took a headline like "Meta strikes $60bn chip deal with AMD...
    (matched: 'strike' from guardian_world)" It matched the word "Strike" which is in war category, but here it's "strike a deal". Another potential problem with just using keywords without context.
        # Ask Bojan to see if there's a way to improve this, making it adaptive instead of hardcoding.
        # Try expanding the keywords for now and see what kind of results we get.

# News sources. 
        # Could include more sources here. Just a few well known ones to start (reuters, bbc, guardian, etc.)

# Crisis score based on weighted average of the following signals:
        # These weights but we need to base these weights on some logic and research. We need to justify the weights. 
        # Ask Marcus to do some research and suggest how to weigh these in a way that makes sense, 

        weights = {
            'volume_spike': 0.30,
            'sentiment_shift': 0.30,
            'keyword_clustering': 0.30,
            'source_diversity': 0.10
        }

# Group by time windows (In detect_unusual_volume function)
        # Grouped into hourly buckets here, 
        # but we could change it to days. 
        # Use Hourly buckets for detecting fast-moving, breaking crises
        # Use Daily buckets for slower-moving, evolving crisis
        # Could make this flexible/adapitve or just stick to one depending on what kind of crises we want to detect.
        
# Check some of the other comments to see logic and interpretation behind score calculation

In [1]:
# crisis_detector
import requests
import feedparser
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from collections import defaultdict
import time
from textblob import TextBlob  # For sentiment analysis

In [ ]:
class CrisisDetector:
    """
    Detects potential crises from real-time news streams
    """
    
    def __init__(self):
        # Crisis keywords by category
        self.crisis_keywords = {
            'election': [
                'election', 'vote', 'ballot', 'campaign', 'polls',
                'presidential', 'democracy', 'voting', 'fraud', 'results'
            ],
            'pandemic': [
                'pandemic', 'epidemic', 'outbreak', 'virus', 'covid',
                'ebola', 'quarantine', 'lockdown', 'vaccine', 'cases',
                'infection', 'hospital', 'icu', 'death'
            ],
            'war_conflict': [
                'war', 'attack', 'strike', 'bombing', 'invasion',
                'military', 'troops', 'rebel', 'missile', 'explosion',
                'conflict', 'violence', 'casualties', 'refugees'
            ],
            'natural_disaster': [
                'earthquake', 'hurricane', 'flood', 'tsunami', 'wildfire',
                'tornado', 'landslide', 'volcano', 'drought', 'storm'
            ],
            'economic': [
                'recession', 'crash', 'crisis', 'inflation', 'bankruptcy',
                'market', 'stock', 'economy', 'debt', 'default',
                'unemployment', 'financial'
            ]
        }
        
        # News sources
        self.rss_feeds = {
            'reuters_world': 'http://feeds.reuters.com/Reuters/worldNews',
            'reuters_top': 'http://feeds.reuters.com/reuters/topNews',
            'bbc_world': 'http://feeds.bbci.co.uk/news/world/rss.xml',
            'guardian_world': 'https://www.theguardian.com/world/rss',
            'ap_news': 'https://feeds.feedburner.com/AssociatedPressTopNews'
        }
        
        # Initialize tracking
        self.news_history = []
        self.crisis_mode = False
        self.current_crisis_type = 'none'
        self.crisis_score = 0.0
        
    def fetch_news_stream(self, hours_back=6):
        """Fetch recent news to create the 'news_stream' input"""
        articles = []
        cutoff_time = datetime.now() - timedelta(hours=hours_back)
        
        for source_name, feed_url in self.rss_feeds.items():
            try:
                feed = feedparser.parse(feed_url)
                for entry in feed.entries[:20]:
                    if hasattr(entry, 'published_parsed'):
                        pub_time = datetime(*entry.published_parsed[:6])
                    else:
                        pub_time = datetime.now()
                    
                    if pub_time < cutoff_time:
                        continue
                    
                    articles.append({
                        'source': source_name,
                        'title': entry.title,
                        'summary': entry.get('summary', ''),
                        'published': pub_time,
                        'content': entry.title + " " + entry.get('summary', '')
                    })
            except Exception as e:
                print(f"Error fetching {source_name}: {e}")
        
        return articles
    
    def calculate_crisis_score(self, news_stream):
        """
        Input: Real-time news headlines + metadata
        Output: Crisis score (0-1), Crisis type, and detailed explanations
        """
        if not news_stream:
            return {
                'crisis_score': 0.0,
                'crisis_type': 'none',
                'crisis_mode': False,
                'adaptive_threshold': 0.5,
                'signals': {},
                'signal_weights': {},
                'keyword_details': {'top_keywords': [], 'all_counts': {}},
                'sample_headlines': [],
                'explanations': ['No news data available'],
                'article_count': 0
            }
        
        # Calculate each signal
        signals = {
            'volume_spike': self.detect_unusual_volume(news_stream),
            'sentiment_shift': self.track_sentiment_over_time(news_stream),
            'keyword_clustering': self.detect_crisis_keywords(news_stream),
            'source_diversity': self.calculate_source_diversity(news_stream)
        }
        
        # Weighted average
        weights = {
            'volume_spike': 0.30,
            'sentiment_shift': 0.30,
            'keyword_clustering': 0.30,
            'source_diversity': 0.10
        }
        
        crisis_score = sum(signals[s] * weights[s] for s in signals)
        
        # Determine crisis type with keyword analysis
        crisis_type, keyword_details = self.determine_crisis_type_with_keywords(news_stream)
        
        # Get sample headlines
        sample_headlines = self.get_sample_headlines(news_stream, crisis_type, n=5)
        
        # Generate explanations
        explanations = self.generate_explanations(signals, weights, crisis_score, crisis_type)
        
        # Determine adaptive threshold based on crisis score
        if crisis_score > 0.7:
            adaptive_threshold = 0.3
            crisis_mode = True
        elif crisis_score > 0.4:
            adaptive_threshold = 0.4
            crisis_mode = True
        else:
            adaptive_threshold = 0.5
            crisis_mode = False
        
        # Store current state
        self.crisis_score = crisis_score
        self.current_crisis_type = crisis_type
        self.crisis_mode = crisis_mode
        self.adaptive_threshold = adaptive_threshold
        
        return {
            'crisis_score': crisis_score,
            'crisis_type': crisis_type,
            'crisis_mode': crisis_mode,
            'adaptive_threshold': adaptive_threshold,
            'signals': signals,
            'signal_weights': weights,
            'keyword_details': keyword_details,
            'sample_headlines': sample_headlines,
            'explanations': explanations,
            'article_count': len(news_stream)
        }
    
    # ============================================
    # SIGNAL IMPLEMENTATIONS
    # ============================================
    
    def detect_unusual_volume(self, news_stream):
        """Signal 1: Detect volume spikes compared to baseline"""
        if not news_stream:
            return 0.0
        
        # Group by time windows (hourly here)
        now = datetime.now()
        articles_by_hour = defaultdict(int)
        
        for article in news_stream:
            hour_key = article['published'].strftime('%Y-%m-%d %H:00')
            articles_by_hour[hour_key] += 1
        
        # Get current hour volume
        current_hour = now.strftime('%Y-%m-%d %H:00')
        current_volume = articles_by_hour.get(current_hour, 0)
        
        # Calculate average volume from previous hours (this is our baseline)
        previous_hours = [h for h in articles_by_hour.keys() if h != current_hour]
        if previous_hours:
            avg_volume = np.mean([articles_by_hour[h] for h in previous_hours])
        else:
            avg_volume = 1
        
        # Calculate spike ratio 
        # (Interpret as: if Spike_Ratio=4.5 then the current news volume is 4.5 times higher than normal/baseline)
        if avg_volume > 0:
            spike_ratio = current_volume / avg_volume
        else:
            spike_ratio = 1
        

        # Normalize to 0-1 (spike_ratio of 3 = 1.0)
        # Examples: If Spike Ratio=1, then volume score=1/3=0.33--> Normal mode
        # Examples: If Spike Ratio=2, then volume score=2/3=0.66--> Maybe something's happening
        # Examples: If Spike Ratio=3 or higher, then volume score=3/3=1 or 5/3 would also output 1 because we'll cap it--> Crisis mode
        # Capping at 1.0 to prevent outliers from dominating, A 10x spike shouldn't score much more than 3x spike since both would point towards a crisis.
        # The /3 threshold is tunable, reduce it make our model more sensitive or increase to make it more conservative

        volume_score = min(spike_ratio / 3, 1.0)
        return volume_score
    
    def track_sentiment_over_time(self, news_stream):
        """Signal 2: Track sentiment shifts (negative sentiment = crisis signal)"""
        if not news_stream:
            return 0.0
        
        # Calculate sentiment for recent articles
        sentiments = []
        for article in news_stream[:50]:
            try:
                blob = TextBlob(article['title'])
                sentiment = blob.sentiment.polarity # -1 to 1
                sentiments.append(sentiment)
            except:
                continue
        
        if not sentiments:
            return 0.0
        
        # Average sentiment (more negative = higher crisis score)
        avg_sentiment = np.mean(sentiments)

        # Convert sentiment to crisis signal
        # -1 (very negative) --> 1.0 (high crisis)
        # 0 (neutral) --> 0.5
        # 1 (positive) --> 0.0 
        sentiment_score = 0.5 - (avg_sentiment * 0.5)

        #To ensure our output stays within [0,1] range 
        # although textblob sentiment shouldn't give an issue being bw. -1 and 1, so could remove this
        sentiment_score = max(0, min(1, sentiment_score))
        
        return sentiment_score
    
    def detect_crisis_keywords(self, news_stream):
        """Signal 3: Detect crisis-related keywords"""
        if not news_stream:
            return 0.0
        
        # Count keyword occurrences by category
        keyword_counts = defaultdict(int)
        total_articles = len(news_stream)
        
        for article in news_stream:
            content = article['title'].lower() + " " + article.get('summary', '').lower()
            
            for category, keywords in self.crisis_keywords.items():
                for keyword in keywords:
                    if keyword in content:
                        keyword_counts[category] += 1
                        break # Count article once per category
        
        # Calculate keyword density score
        if total_articles > 0:
            total_keyword_articles = sum(keyword_counts.values())
            keyword_score = min(total_keyword_articles / total_articles, 1.0)
        else:
            keyword_score = 0.0
        
        return keyword_score
    
    def calculate_source_diversity(self, news_stream):
        """Signal 4: Are many different sources reporting the same event?"""
        if not news_stream:
            return 0.0
        
        # Count unique sources (Need to expand on our sources to get better results from this)
        sources = set(article['source'] for article in news_stream)
        unique_sources = len(sources)

        # More sources reporting --> higher confidence in event being a crisis
        # 5+ sources = 1.0, 1 source = 0.2 
        # Again we might want to adjust this by figuring out minimum number of reliable sources that need to be reporting on the same event to count as a crisis

        diversity_score = min(unique_sources / 5, 1.0)
        
        return diversity_score
    
    def determine_crisis_type(self, news_stream):
        """Determine the primary crisis type based on keyword frequency"""
        if not news_stream:
            return 'none'
        
        category_counts = defaultdict(int)
        
        for article in news_stream:
            content = article['title'].lower() + " " + article.get('summary', '').lower()
            
            for category, keywords in self.crisis_keywords.items():
                for keyword in keywords:
                    if keyword in content:
                        category_counts[category] += 1
                        break
        
        if not category_counts:
            return 'none'
        
        # Return category with highest count
        return max(category_counts.items(), key=lambda x: x[1])[0]
    
    def determine_crisis_type_with_keywords(self, news_stream):
        """
        Enhanced crisis type detection with keyword frequency analysis
        """
        from collections import defaultdict
        
        category_counts = defaultdict(int)
        keyword_hits = defaultdict(list)
        
        for article in news_stream:
            content = article['title'].lower() + " " + article.get('summary', '').lower()
            
            for category, keywords in self.crisis_keywords.items():
                for keyword in keywords:
                    if keyword in content:
                        category_counts[category] += 1
                        keyword_hits[category].append({
                            'keyword': keyword,
                            'title': article['title']
                        })
                        break
        
        if not category_counts:
            return 'none', {'top_keywords': [], 'all_counts': {}}
        
        crisis_type = max(category_counts.items(), key=lambda x: x[1])[0]
        
        crisis_keywords = keyword_hits.get(crisis_type, [])
        keyword_freq = defaultdict(int)
        for hit in crisis_keywords:
            keyword_freq[hit['keyword']] += 1
        
        top_keywords = sorted(keyword_freq.items(), key=lambda x: x[1], reverse=True)[:5]
        
        return crisis_type, {
            'top_keywords': top_keywords,
            'total_mentions': category_counts[crisis_type],
            'all_counts': dict(category_counts)
        }
    
    def generate_explanations(self, signals, weights, crisis_score, crisis_type):
        """
        Generate human-readable explanations for each signal
        """
        explanations = []
        
        if signals['volume_spike'] > 0.7:
            explanations.append(f"📈 HIGH VOLUME: {signals['volume_spike']*100:.0f}% spike in news activity")
        elif signals['volume_spike'] > 0.3:
            explanations.append(f"📈 Elevated news volume ({signals['volume_spike']:.2f})")
        
        if signals['sentiment_shift'] > 0.7:
            explanations.append(f"😠 VERY NEGATIVE sentiment detected")
        elif signals['sentiment_shift'] > 0.5:
            explanations.append(f"😐 Negative sentiment shift ({signals['sentiment_shift']:.2f})")
        
        if signals['keyword_clustering'] > 0.5:
            explanations.append(f"🔍 Strong {crisis_type} keyword presence")
        
        if signals['source_diversity'] > 0.6:
            explanations.append(f"🌐 Multiple sources reporting ({signals['source_diversity']*100:.0f}% diversity)")
        
        explanations.append(f"📊 Final crisis score: {crisis_score:.2f} (weighted combination)")
        
        return explanations
    
    def get_sample_headlines(self, news_stream, crisis_type, n=5):
        """
        Get sample headlines that triggered the crisis detection
        """
        samples = []
        keywords = self.crisis_keywords.get(crisis_type, [])
        
        for article in news_stream:
            content = article['title'].lower()
            for keyword in keywords:
                if keyword in content:
                    samples.append({
                        'title': article['title'],
                        'keyword_matched': keyword,
                        'source': article['source']
                    })
                    break
            if len(samples) >= n:
                break
        
        return samples
    
    def monitor_continuously(self, interval_minutes=15):
        """
        Continuous monitoring loop that updates crisis state
        """
        print("Starting crisis monitor...")
        
        while True:
            try:
                news_stream = self.fetch_news_stream(hours_back=6)
                result = self.calculate_crisis_score(news_stream)
                
                print(f"\n{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"🚨 Crisis Mode: {result['crisis_mode']}")
                print(f"📊 Crisis Score: {result['crisis_score']:.3f}")
                print(f"📍 Crisis Type: {result['crisis_type']}")
                print(f"⚖️ Adaptive Threshold: {result['adaptive_threshold']}")
                print(f"\nSignals:")
                for signal, value in result['signals'].items():
                    print(f"  • {signal}: {value:.3f}")
                
                self.news_history.append({
                    'timestamp': datetime.now(),
                    'result': result
                })
                
                cutoff = datetime.now() - timedelta(hours=24)
                self.news_history = [
                    h for h in self.news_history 
                    if h['timestamp'] > cutoff
                ]
                
                time.sleep(interval_minutes * 60)
                
            except KeyboardInterrupt:
                print("\nMonitoring stopped.")
                break
            except Exception as e:
                print(f"Error: {e}")
                time.sleep(60)

In [4]:
if __name__ == "__main__":
    detector = CrisisDetector()
    
    # Fetch news
    news_stream = detector.fetch_news_stream(hours_back=6)
    print(f"📰 Fetched {len(news_stream)} articles")
    
    if news_stream:
        result = detector.calculate_crisis_score(news_stream)
        
        print("\n" + "="*60)
        print("CRISIS DETECTION RESULTS")
        print("="*60)
        
        # Core results
        print(f"\n📍 Crisis Type: {result['crisis_type'].upper()}")
        print(f"📊 Crisis Score: {result['crisis_score']:.3f}")
        print(f"🚨 Crisis Mode: {result['crisis_mode']}")
        print(f"⚖️ Adaptive Threshold: {result['adaptive_threshold']}")
        
        # Keyword details
        if result['keyword_details']['top_keywords']:
            print(f"\n🔍 Top Keywords Detected:")
            for keyword, count in result['keyword_details']['top_keywords']:
                print(f"  • '{keyword}': {count} mentions")
        
        # Signal breakdown
        print(f"\n📈 Signal Breakdown:")
        for signal, value in result['signals'].items():
            weight = result['signal_weights'][signal]
            bar = "█" * int(value * 20)
            print(f"  {signal:20} [{bar:20}] {value:.3f} (weight: {weight:.2f})")
        
        # Explanations
        print(f"\n💡 Explanations:")
        for exp in result['explanations']:
            print(f"  • {exp}")
        
        # Sample headlines
        if result['sample_headlines']:
            print(f"\n📰 Sample Headlines:")
            for headline in result['sample_headlines']:
                print(f"  • {headline['title'][:80]}...")
                print(f"    (matched: '{headline['keyword_matched']}' from {headline['source']})")
    else:
        print("❌ No articles fetched")

📰 Fetched 14 articles

CRISIS DETECTION RESULTS

📍 Crisis Type: WAR_CONFLICT
📊 Crisis Score: 0.352
🚨 Crisis Mode: False
⚖️ Adaptive Threshold: 0.5

🔍 Top Keywords Detected:
  • 'violence': 1 mentions
  • 'explosion': 1 mentions
  • 'war': 1 mentions

📈 Signal Breakdown:
  volume_spike         [                    ] 0.000 (weight: 0.30)
  sentiment_shift      [█████████           ] 0.467 (weight: 0.30)
  keyword_clustering   [███████████         ] 0.571 (weight: 0.30)
  source_diversity     [████████            ] 0.400 (weight: 0.10)

💡 Explanations:
  • 🔍 Strong war_conflict keyword presence
  • 📊 Final crisis score: 0.35 (weighted combination)

📰 Sample Headlines:
  • Mexico hunts 23 inmates sprung from jail during wave of violence...
    (matched: 'violence' from bbc_world)
  • Moscow rail hub explosion was suicide bomb, Russian officials say...
    (matched: 'explosion' from bbc_world)
  • Trump’s new global tariffs kick in at 10%; Meta strikes $60bn chip deal with AMD...
    (match